# 🩺 Complete Liver Disease Prediction Project
**Stack:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn, Imbalanced-Learn (SMOTE), Joblib

---

## 📌 Project Objectives & Overview
Liver disease is a leading cause of global mortality, frequently remaining asymptomatic until reaching critical hepatic impairment. Early clinical detection through routine **Liver Function Tests (LFTs)** is crucial.

In this notebook, we implement the complete Machine Learning lifecycle:
1. **Phase 1: Dataset Analysis & Discovery** – Automatic file detection, encoding resolution, column sanitization, missing value & duplicate audit.
2. **Phase 2: Preprocessing & Leakage Prevention** – Stratified partitioning, Scikit-Learn `ColumnTransformer` (median imputation, scaling, one-hot encoding), and SMOTE within `imblearn.pipeline.Pipeline`.
3. **Phase 3: Model Training & Benchmark** – Comparing **Logistic Regression**, **Decision Tree**, and **Random Forest**.
4. **Phase 4: Model Persistence & Clinical Inference** – Joblib pipeline serialization, domain input validation (Direct $\le$ Total Bilirubin), and calibrated risk scoring.
5. **Phase 5: Exploratory Data Analysis (EDA)** – Target distributions, correlation heatmaps, and biomarker distribution curves.


## 📦 1. Environment Setup & Library Imports
We import essential libraries for data manipulation, visualization, pipeline construction, and evaluation.


In [ ]:
import os
import re
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

print("All dependencies successfully imported!")


---
## 🔍 2. Phase 1: Automated Dataset Discovery & Inspection
We automatically locate the dataset and handle character encodings (e.g. `latin1` to handle raw non-breaking spaces `\xa0` often found in clinical datasets).


In [ ]:
def find_dataset():
    candidates = [
        os.path.join("dataset", "liver_disease_data.csv"),
        os.path.join("dataset", "Liver Patient Dataset (LPD)_train.csv"),
        "Liver Patient Dataset (LPD)_train.csv",
        "../dataset/liver_disease_data.csv",
        "../dataset/Liver Patient Dataset (LPD)_train.csv"
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("Liver patient dataset not found.")

dataset_path = find_dataset()
print(f"Dataset located at: {dataset_path}")

# Load using latin1 to safely handle non-breaking space characters
try:
    df_raw = pd.read_csv(dataset_path, encoding='latin1')
except Exception:
    df_raw = pd.read_csv(dataset_path, encoding='utf-8')

print(f"Dataset Dimensions: {df_raw.shape[0]:,} Rows, {df_raw.shape[1]} Columns")
print("\nRaw Column Names:")
for i, col in enumerate(df_raw.columns):
    print(f"  [{i}] {repr(col)} ({df_raw[col].dtype})")

df_raw.head()


### 🧹 Column Name Sanitization & Target Mapping
Clinical datasets often contain non-breaking spaces (`\xa0`), mixed casing, and typographical variations (e.g., `Total Protiens`).
We standardize all column names into clean, lowercase snake_case identifiers.

**Target Mapping:**
- Original dataset encoding: `1` = Liver Disease Patient, `2` = Healthy Patient.
- We map: `1` $\rightarrow$ **1** (Positive Class: Liver Disease), `2` $\rightarrow$ **0** (Negative Class: Healthy).
This allows standard evaluation metrics (Sensitivity / Recall, ROC-AUC) to reflect disease detection accurately.


In [ ]:
RAW_COLUMN_MAPPING = {
    "age of the patient": "age",
    "gender of the patient": "gender",
    "total bilirubin": "total_bilirubin",
    "direct bilirubin": "direct_bilirubin",
    "alkphos alkaline phosphotase": "alkaline_phosphotase",
    "sgpt alamine aminotransferase": "alamine_aminotransferase",
    "sgot aspartate aminotransferase": "aspartate_aminotransferase",
    "total protiens": "total_proteins",
    "total proteins": "total_proteins",
    "alb albumin": "albumin",
    "a/g ratio albumin and globulin ratio": "albumin_globulin_ratio",
    "result": "result"
}

def clean_col(c):
    cleaned = c.replace('\xa0', ' ').strip().lower()
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return RAW_COLUMN_MAPPING.get(cleaned, cleaned)

df = df_raw.copy()
df.columns = [clean_col(c) for c in df.columns]

# Clean gender strings
if 'gender' in df.columns:
    df['gender'] = df['gender'].astype(str).str.strip().str.title()
    df['gender'] = df['gender'].replace({'Nan': np.nan, 'None': np.nan, '': np.nan})

# Map target: 1 -> 1 (Diseased), 2 -> 0 (Healthy)
df = df[df['result'].isin([1, 2])].reset_index(drop=True)
df['result'] = df['result'].map({1: 1, 2: 0}).astype(int)

print("Standardized Columns:", list(df.columns))
df.head()


### 🔍 Missing Values & Duplicate Records Analysis
Let us examine the extent of missing values and check for duplicate patient records.


In [ ]:
print("--- MISSING VALUES AUDIT ---")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({"Missing Count": missing, "Percentage (%)": missing_pct})
print(missing_df[missing_df["Missing Count"] > 0])

print("\n--- DUPLICATE ROWS AUDIT ---")
dup_count = df.duplicated().sum()
print(f"Total Duplicate Rows: {dup_count:,} ({dup_count / len(df) * 100:.2f}%)")

# Deduplication to prevent data leakage between train and test splits
df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"Dataset Size After Deduplication: {len(df_clean):,} unique patient observations")

print("\n--- TARGET CLASS DISTRIBUTION (Deduplicated) ---")
counts = df_clean['result'].value_counts()
props = df_clean['result'].value_counts(normalize=True) * 100
for cls, count in counts.items():
    label = "Liver Disease (1)" if cls == 1 else "Healthy (0)"
    print(f"  {label}: {count:,} patients ({props[cls]:.2f}%)")


---
## 📊 3. Phase 5: Exploratory Data Analysis (EDA)
We visualize the target distribution, biomarker correlation structure, and feature distributions.


In [ ]:
# 1. Target Class Distribution (Bar + Donut Chart)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

labels = ["Liver Disease (1)", "Healthy (0)"]
counts = [df_clean['result'].value_counts()[1], df_clean['result'].value_counts()[0]]
colors = ["#e74c3c", "#2ecc71"]

bars = ax1.bar(labels, counts, color=colors, width=0.5, edgecolor="black", alpha=0.85)
ax1.set_title("Target Class Counts", fontsize=13, fontweight="bold")
ax1.set_ylabel("Number of Patients", fontsize=11)
ax1.grid(axis="y", linestyle="--", alpha=0.7)
for bar in bars:
    h = bar.get_height()
    pct = (h / len(df_clean)) * 100
    ax1.annotate(f"{h:,}\n({pct:.1f}%)", (bar.get_x() + bar.get_width() / 2, h / 2),
                 ha="center", va="center", color="white", fontweight="bold", fontsize=11)

ax2.pie(counts, labels=labels, colors=colors, autopct="%1.1f%%", startangle=140,
        wedgeprops=dict(width=0.4, edgecolor="w", linewidth=2))
ax2.set_title("Class Proportion Breakdown", fontsize=13, fontweight="bold")

plt.suptitle("Target Class Distribution: Indian Liver Patient Dataset", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 2. Correlation Heatmap for Numerical Biomarkers
numerical_features = [
    "age", "total_bilirubin", "direct_bilirubin", "alkaline_phosphotase",
    "alamine_aminotransferase", "aspartate_aminotransferase",
    "total_proteins", "albumin", "albumin_globulin_ratio"
]

num_cols_with_target = numerical_features + ['result']
corr = df_clean[num_cols_with_target].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax
)
ax.set_title("Correlation Matrix of Hepatic Biomarkers & Diagnosis", fontsize=13, fontweight="bold", pad=15)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# 3. Biomarker Distributions Conditioned on Diagnosis
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, col in enumerate(numerical_features):
    ax = axes[i]
    sns.kdeplot(
        data=df_clean, x=col, hue='result', common_norm=False,
        palette={1: "#e74c3c", 0: "#2ecc71"}, ax=ax, fill=True, alpha=0.3
    )
    ax.set_title(f"Distribution: {col}", fontsize=11, fontweight="bold")
    ax.set_xlabel(col, fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    if df_clean[col].max() > 100:
        ax.set_xscale("log")
        ax.set_xlabel(f"{col} (log scale)", fontsize=9)

plt.suptitle("Feature Distributions by Diagnosis (0: Healthy, 1: Diseased)",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()


---
## ⚙️ 4. Phase 2: Data Preprocessing & Pipeline Construction

### Preventing Data Leakage
1. **Deduplication:** Duplicated samples are removed prior to train-test splitting.
2. **Stratified Split:** An 80/20 train/test split maintains the 71.3% / 28.7% class ratio in both sets.
3. **Pipeline Encapsulation:** Imputation medians and scaling parameters are computed strictly on the training partition.
4. **SMOTE inside `imblearn.pipeline.Pipeline`:** Resampling is applied strictly to the training fold during `fit()`, never to test or prediction data.


In [ ]:
FEATURE_NAMES = [
    "age", "gender", "total_bilirubin", "direct_bilirubin",
    "alkaline_phosphotase", "alamine_aminotransferase",
    "aspartate_aminotransferase", "total_proteins", "albumin",
    "albumin_globulin_ratio"
]

NUMERICAL_FEATURES = [
    "age", "total_bilirubin", "direct_bilirubin", "alkaline_phosphotase",
    "alamine_aminotransferase", "aspartate_aminotransferase",
    "total_proteins", "albumin", "albumin_globulin_ratio"
]

CATEGORICAL_FEATURES = ["gender"]

X = df_clean[FEATURE_NAMES].copy()
y = df_clean['result'].copy()

# Stratified Train/Test Split (80% Train, 20% Held-Out Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Training Cohort: {X_train.shape[0]:,} samples")
print(f"Held-Out Test Cohort: {X_test.shape[0]:,} samples")
print(f"Train Disease Prevalence: {y_train.mean()*100:.2f}% | Test Disease Prevalence: {y_test.mean()*100:.2f}%")


In [ ]:
def build_preprocessor():
    num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    
    cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"))
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", num_pipeline, NUMERICAL_FEATURES),
            ("cat", cat_pipeline, CATEGORICAL_FEATURES)
        ],
        remainder="drop"
    )
    return preprocessor

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(X_train)
print(f"Transformed Training Feature Matrix Shape: {X_train_transformed.shape}")
print("ColumnTransformer successfully constructed and verified!")


---
## 🤖 5. Phase 3: Model Training, Evaluation & Comparison
We train and benchmark three distinct machine learning algorithms:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree Classifier** (Non-linear rule-based tree)
3. **Random Forest Classifier** (Ensemble of decorrelated decision trees)

Each model is evaluated on the held-out test set using:
- **Accuracy, Precision, Recall (Sensitivity), F1-Score, ROC-AUC, Specificity**
- **Confusion Matrix & Classification Report**


In [ ]:
candidate_models = {
    "Logistic Regression": LogisticRegression(max_iter=1500, C=1.0, solver="lbfgs", random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, min_samples_split=20, min_samples_leaf=10, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=12, min_samples_split=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
}

trained_pipelines = {}
metrics_summary = {}

print("="*65)
print("TRAINING CANDIDATE MODELS (SMOTE APPLIED ON TRAINING SPLIT)")
print("="*65)

for name, model in candidate_models.items():
    print(f"\n---> Training: {name}...")
    pipeline = ImbPipeline([
        ("preprocessor", build_preprocessor()),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else 0.0
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    
    metrics_summary[name] = {
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall (Sensitivity)": round(rec, 4),
        "F1-Score": round(f1, 4),
        "ROC-AUC": round(roc_auc, 4),
        "Specificity": round(spec, 4),
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
        "Confusion_Matrix": cm.tolist()
    }
    trained_pipelines[name] = pipeline
    
    print(f"  Accuracy:    {acc*100:.2f}%")
    print(f"  Precision:   {prec*100:.2f}%")
    print(f"  Recall:      {rec*100:.2f}% (Sensitivity)")
    print(f"  F1-Score:    {f1:.4f}")
    print(f"  ROC-AUC:     {roc_auc:.4f}")
    print(f"  Specificity: {spec*100:.2f}%")


In [ ]:
# Benchmark Comparison Table
df_benchmark = pd.DataFrame(metrics_summary).T[
    ["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC", "Specificity"]
]
print("\n" + "="*50)
print("TEST SET BENCHMARK TABLE")
print("="*50)
df_benchmark


### 📈 Model Performance Visualizations
We plot multi-metric bar charts, side-by-side confusion matrices, and ROC curves.


In [ ]:
# 1. Multi-Metric Comparison Bar Chart
fig, ax = plt.subplots(figsize=(11, 5))
df_benchmark[["Accuracy", "Precision", "Recall (Sensitivity)", "F1-Score", "ROC-AUC"]].plot(
    kind="bar", ax=ax, width=0.8, colormap="viridis"
)
ax.set_title("Candidate Model Performance Comparison on Held-Out Test Set", fontsize=13, fontweight="bold", pad=15)
ax.set_ylabel("Score (0.0 to 1.0)", fontsize=11)
ax.set_xlabel("Model Architecture", fontsize=11)
ax.set_ylim(0, 1.1)
ax.tick_params(axis="x", rotation=0, labelsize=10)
ax.legend(loc="lower right", frameon=True)

for p in ax.patches:
    height = p.get_height()
    if height > 0.05:
        ax.annotate(f"{height:.2f}", (p.get_x() + p.get_width() / 2., height),
                    ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Confusion Matrices Comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, metrics) in zip(axes, metrics_summary.items()):
    cm = np.array(metrics["Confusion_Matrix"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
        xticklabels=["Healthy (0)", "Liver Disease (1)"],
        yticklabels=["Healthy (0)", "Liver Disease (1)"]
    )
    ax.set_title(f"{name}\nRecall: {metrics['Recall (Sensitivity)']*100:.1f}% | Spec: {metrics['Specificity']*100:.1f}%",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted Diagnosis", fontsize=10)
    ax.set_ylabel("Actual Diagnosis", fontsize=10)

plt.suptitle("Confusion Matrices (Held-Out Test Cohort: 3,874 Patients)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# 3. Receiver Operating Characteristic (ROC) Curves
fig, ax = plt.subplots(figsize=(8, 6))

for name, pipeline in trained_pipelines.items():
    if hasattr(pipeline.named_steps["classifier"], "predict_proba"):
        y_proba = pipeline.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        auc = roc_auc_score(y_test, y_proba)
        ax.plot(fpr, tpr, lw=2, label=f"{name} (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], color="grey", lw=1.5, linestyle="--", label="Chance (AUC = 0.500)")
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
ax.set_ylabel("True Positive Rate (Recall / Sensitivity)", fontsize=11)
ax.set_title("Receiver Operating Characteristic (ROC) Curves", fontsize=13, fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()


---
## 💾 6. Phase 4: Model Selection & Persistence

### Clinical Selection Rationale
In clinical screening, we formulate a composite score prioritizing **Recall** (catching sick individuals) and **F1-Score**:
$$\text{Composite Score} = 0.40 \times \text{Recall} + 0.35 \times \text{F1-Score} + 0.25 \times \text{ROC-AUC}$$


In [ ]:
best_model_name = None
best_composite_score = -1.0

print("COMPOSITE CLINICAL SCORING:")
for name, m in metrics_summary.items():
    comp = (0.40 * m["Recall (Sensitivity)"]) + (0.35 * m["F1-Score"]) + (0.25 * m["ROC-AUC"])
    print(f"  {name:20s}: Composite Score = {comp:.4f} (Recall: {m['Recall (Sensitivity)']:.4f}, F1: {m['F1-Score']:.4f}, AUC: {m['ROC-AUC']:.4f})")
    if comp > best_composite_score:
        best_composite_score = comp
        best_model_name = name

print(f"\n>>> Selected Optimal Model: {best_model_name} (Composite Score: {best_composite_score:.4f})")

# Save complete fitted pipeline using Joblib
os.makedirs("models", exist_ok=True)
model_path = os.path.join("models", "best_liver_model.joblib")
best_pipeline = trained_pipelines[best_model_name]
joblib.dump(best_pipeline, model_path)
print(f"[OK] Complete fitted pipeline successfully serialized to: {model_path}")

# Save metadata
metadata = {
    "best_model_name": best_model_name,
    "features": FEATURE_NAMES,
    "metrics_summary": metrics_summary,
    "best_model_metrics": metrics_summary[best_model_name],
    "class_mapping": {"0": "Healthy", "1": "Liver Disease Detected"}
}
with open(os.path.join("models", "model_metadata.json"), "w") as f:
    json.dump(metadata, f, indent=4)
print("[OK] Metadata saved to models/model_metadata.json")


---
## 🧪 7. Reusable Clinical Prediction Engine with Validation
We implement reusable inference functions with:
1. **Schema & key sanitization**
2. **Plausible physiological range checks**
3. **Medical domain logic check:** Direct Bilirubin must not exceed Total Bilirubin
4. **Risk tier categorization & Medical Disclaimer**


In [ ]:
PHYSIOLOGICAL_RANGES = {
    "age": {"min": 1, "max": 120, "label": "Age (years)"},
    "gender": {"allowed": ["Male", "Female"], "label": "Biological Sex"},
    "total_bilirubin": {"min": 0.1, "max": 80.0, "label": "Total Bilirubin (mg/dL)"},
    "direct_bilirubin": {"min": 0.05, "max": 30.0, "label": "Direct Bilirubin (mg/dL)"},
    "alkaline_phosphotase": {"min": 20.0, "max": 3000.0, "label": "Alkaline Phosphatase (ALP IU/L)"},
    "alamine_aminotransferase": {"min": 5.0, "max": 3000.0, "label": "ALT / SGPT (IU/L)"},
    "aspartate_aminotransferase": {"min": 5.0, "max": 5000.0, "label": "AST / SGOT (IU/L)"},
    "total_proteins": {"min": 1.0, "max": 15.0, "label": "Total Proteins (g/dL)"},
    "albumin": {"min": 0.5, "max": 8.0, "label": "Albumin (g/dL)"},
    "albumin_globulin_ratio": {"min": 0.1, "max": 5.0, "label": "A/G Ratio"}
}

def validate_input(patient_data):
    errors = []
    for f in FEATURE_NAMES:
        if f not in patient_data or patient_data[f] is None:
            errors.append(f"Missing required biomarker field: '{f}'")
    if errors:
        return False, errors
        
    # Check gender
    gender = str(patient_data['gender']).strip().title()
    if gender not in ["Male", "Female"]:
        errors.append(f"Invalid gender '{patient_data['gender']}'. Must be 'Male' or 'Female'.")
        
    # Check numerical bounds
    for feat, limits in PHYSIOLOGICAL_RANGES.items():
        if feat == "gender":
            continue
        try:
            val = float(patient_data[feat])
            if val < limits["min"] or val > limits["max"]:
                errors.append(f"{limits['label']} ({val}) is outside plausible range [{limits['min']}, {limits['max']}].")
        except (ValueError, TypeError):
            errors.append(f"'{feat}' must be numeric.")
            
    # Domain check: Direct Bilirubin <= Total Bilirubin
    try:
        tb = float(patient_data.get("total_bilirubin", 0))
        db = float(patient_data.get("direct_bilirubin", 0))
        if db > tb + 0.1:
            errors.append(f"Direct Bilirubin ({db} mg/dL) cannot exceed Total Bilirubin ({tb} mg/dL).")
    except Exception:
        pass
        
    return len(errors) == 0, errors

def predict_patient(patient_dict, model_file=model_path):
    model = joblib.load(model_file)
    is_valid, errors = validate_input(patient_dict)
    if not is_valid:
        return {"success": False, "errors": errors}
        
    df_single = pd.DataFrame([{f: patient_dict[f] for f in FEATURE_NAMES}])
    df_single["gender"] = df_single["gender"].astype(str).str.strip().str.title()
    for col in NUMERICAL_FEATURES:
        df_single[col] = pd.to_numeric(df_single[col], errors="coerce")
        
    pred = int(model.predict(df_single)[0])
    prob = float(model.predict_proba(df_single)[0][1]) if hasattr(model, "predict_proba") else None
    
    if prob is not None:
        if prob < 0.30:
            risk = "Low Risk"
        elif prob < 0.60:
            risk = "Moderate / Borderline Risk"
        else:
            risk = "High Risk"
    else:
        risk = "Elevated" if pred == 1 else "Normal"
        
    label = "Liver Disease Detected" if pred == 1 else "Healthy / No Liver Disease Detected"
    
    return {
        "success": True,
        "prediction": pred,
        "label": label,
        "probability": round(prob, 4) if prob else None,
        "risk_level": risk,
        "disclaimer": "[NOTICE]: This is an experimental machine learning estimate for educational purposes, not a medical diagnosis."
    }

print("Prediction engine ready!")


### 🧪 Live Testing & Verification
We test the prediction pipeline with:
1. **Case 1:** Elevated hepatic biomarkers (Suspected Liver Disease)
2. **Case 2:** Normal physiological biomarkers (Healthy Profile)
3. **Case 3:** Invalid Input (Direct Bilirubin > Total Bilirubin, Impossible Age)


In [ ]:
# Case 1: High-Risk Liver Disease Patient
patient_sick = {
    "age": 62, "gender": "Male",
    "total_bilirubin": 10.9, "direct_bilirubin": 5.5,
    "alkaline_phosphotase": 699, "alamine_aminotransferase": 64,
    "aspartate_aminotransferase": 100, "total_proteins": 7.5,
    "albumin": 3.2, "albumin_globulin_ratio": 0.74
}
res1 = predict_patient(patient_sick)
print("=== CASE 1: SICK PATIENT SAMPLE ===")
print("Predicted Class:", res1["prediction"])
print("Diagnosis Label:", res1["label"])
print(f"Disease Probability: {res1['probability']*100:.2f}%")
print("Risk Category:", res1["risk_level"])
print(res1["disclaimer"])

# Case 2: Healthy Profile Patient
patient_healthy = {
    "age": 28, "gender": "Female",
    "total_bilirubin": 0.7, "direct_bilirubin": 0.1,
    "alkaline_phosphotase": 130, "alamine_aminotransferase": 18,
    "aspartate_aminotransferase": 20, "total_proteins": 7.1,
    "albumin": 4.2, "albumin_globulin_ratio": 1.35
}
res2 = predict_patient(patient_healthy)
print("\n=== CASE 2: HEALTHY PATIENT SAMPLE ===")
print("Predicted Class:", res2["prediction"])
print("Diagnosis Label:", res2["label"])
print(f"Disease Probability: {res2['probability']*100:.2f}%")
print("Risk Category:", res2["risk_level"])

# Case 3: Invalid Input Test
patient_invalid = {
    "age": 195, "gender": "Unknown",
    "total_bilirubin": 1.0, "direct_bilirubin": 8.0  # DB > TB
}
res3 = predict_patient(patient_invalid)
print("\n=== CASE 3: INVALID INPUT TEST ===")
print("Success:", res3["success"])
print("Caught Errors:")
for e in res3["errors"]:
    print(" -", e)




### Key Methodologys:
1. **Preventing Data Leakage:** Removing duplicate rows prior to splitting avoided overly optimistic test scores. SMOTE was applied strictly within training folds using `imblearn.pipeline.Pipeline`.
2. **Median Imputation for Medical Outliers:** Liver enzymes (ALT, AST, ALP) feature highly skewed distributions with clinical outliers exceeding 2,000 IU/L. Using the **Median** rather than Mean prevents distortion.
3. **Metric Selection in Medical ML:** Precision alone is inadequate for medical screening. Optimizing for **Recall (Sensitivity)** ensures actual disease cases are detected, minimizing fatal False Negatives.
4. **Clinical Validation Rules:** Enforcing physiological constraints (e.g. $\text{Direct Bilirubin} \le \text{Total Bilirubin}$) safeguards against illogical model inputs.
